In [61]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [62]:
import time

In [63]:
from torch.profiler import profile, record_function, ProfilerActivity

In [64]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


In [65]:

print("\n" + "="*60)
print("TOPIC 3 — PAGED ATTENTION")
print("="*60)


TOPIC 3 — PAGED ATTENTION


In [66]:
# for paged attention first of all, one thing should come in my mind!
# Each memory page can store KV cache for 4 tokens.
# Page 0 → stores 4 tokens
# Page 1 → stores next 4 tokens
# Page 2 → stores next 4 tokens
# GPU has 16 pages available total.
# page_table = {
#    "req1": [2,3],
#    "req2": [5,6,7]
# }
# page_index → actual KV tensor stored

# Meaning:

# Which KV cache tensor lives inside each page

In [67]:
PAGE_SIZE = 4
TOTAL_PAGES = 16
free_pages = list(range(TOTAL_PAGES))
page_table = {}
page_data = {}
# This is given data

allocating the free to pages for every new page requirement

In [68]:
def allocate_page(request_id,k_token,v_token):
  if not free_pages:
    raise MemoryError("GPU OOM! Out of pages")
  page = free_pages.pop()
  # now the page contains a page which is free
  page_table.setdefault(request_id,[]).append(page)
  page_data[page]={"k":[k_token],"v":[v_token]}
  return page

In [77]:
def add_token_to_page(request_id, k_token, v_token):
  pages = page_table.get(request_id, [])
  if pages:
    last_page = pages[-1]
    tokens_in_page = len(page_data[last_page]["k"])
    if tokens_in_page < PAGE_SIZE:
      page_data[last_page]["k"].append(k_token)
      page_data[last_page]["v"].append(v_token)
    else:
      print(f"  Request {request_id}: page {last_page} full, allocating new page")
      allocate_page(request_id,k_token,v_token)
  else:
    print(f"  Request {request_id}: no pages allocated, allocating new page")
    allocate_page(request_id,k_token,v_token)




In [74]:
def free_request(request_id):
  pages = page_table.pop(request_id, [])
  for page in pages:
    free_pages.append(page)
    del page_data[page]
  return len(pages)

In [75]:
def get_all_kv(request_id):
  all_k = []
  all_v = []
  for page in page_table.get(request_id, []):
    all_k.extend(page_data[page]["k"])
    all_v.extend(page_data[page]["v"])
  return all_k, all_v

In [78]:
# --- SIMULATE 3 CONCURRENT REQUESTS ---
print("\nSimulating 3 concurrent users...\n")

# Request A — 6 tokens (will need 2 pages since PAGE_SIZE=4)
print("Request A generating 6 tokens:")
for token_idx in range(6):
    k = torch.randn(64)    # fake key vector
    v = torch.randn(64)    # fake value vector
    add_token_to_page("user_A", k, v)
print(f"  user_A owns pages: {page_table['user_A']}")
print(f"  Free pages remaining: {len(free_pages)}")

# Request B — 3 tokens (fits in 1 page)
print("\nRequest B generating 3 tokens:")
for token_idx in range(3):
    k = torch.randn(64)
    v = torch.randn(64)
    add_token_to_page("user_B", k, v)
print(f"  user_B owns pages: {page_table['user_B']}")
print(f"  Free pages remaining: {len(free_pages)}")

# Request C — 4 tokens (exactly 1 page)
print("\nRequest C generating 4 tokens:")
for token_idx in range(4):
    k = torch.randn(64)
    v = torch.randn(64)
    add_token_to_page("user_C", k, v)
print(f"  user_C owns pages: {page_table['user_C']}")
print(f"  Free pages remaining: {len(free_pages)}")

# Show memory state
print("\n--- MEMORY STATE ---")
print(f"Total pages: {TOTAL_PAGES}")
print(f"Used pages:  {TOTAL_PAGES - len(free_pages)}")
print(f"Free pages:  {len(free_pages)}")
print(f"user_A pages: {page_table['user_A']} ({sum(len(page_data[p]['k']) for p in page_table['user_A'])} tokens)")
print(f"user_B pages: {page_table['user_B']} ({sum(len(page_data[p]['k']) for p in page_table['user_B'])} tokens)")
print(f"user_C pages: {page_table['user_C']} ({sum(len(page_data[p]['k']) for p in page_table['user_C'])} tokens)")



Simulating 3 concurrent users...

Request A generating 6 tokens:
  Request user_A: page 15 full, allocating new page
  Request user_A: page 14 full, allocating new page
  user_A owns pages: [15, 14, 13]
  Free pages remaining: 13

Request B generating 3 tokens:
  Request user_B: no pages allocated, allocating new page
  user_B owns pages: [12]
  Free pages remaining: 12

Request C generating 4 tokens:
  Request user_C: no pages allocated, allocating new page
  user_C owns pages: [11]
  Free pages remaining: 11

--- MEMORY STATE ---
Total pages: 16
Used pages:  5
Free pages:  11
user_A pages: [15, 14, 13] (9 tokens)
user_B pages: [12] (3 tokens)
user_C pages: [11] (4 tokens)


# continuous batching

In [80]:
import queue, time

request_queue = queue.Queue()
# let us create a queue for requests to wait
batch = []
# intitally the processing batch is empty
MAX_BATCH = 4
# the maximum num of req each batch can accomodate is 4

def process_batch(batch):
    # this generates a forward pass
    print(f"Processing batch of {len(batch)} requests simultaneously")
    time.sleep(0.1)  # simulate model forward pass
    # suppose gpu takes 0.1 seconds to run a pass

# Simulate continuous batching
for i in range(10):
    request_queue.put(f"request_{i}")
    # lets create 10 requesrs are waiting in queue

while not request_queue.empty() or batch:
    # run the loop until the batch is not empty and the waiting queue is not empty
    while len(batch) < MAX_BATCH and not request_queue.empty():
        # when lenth of the batch is less than the max count  and queue still has numbers
        batch.append(request_queue.get())
        # then take a request from  queue and append it to the batch
                # fill batch with new requests
    process_batch(batch)
    # and now process the batch
    batch.pop(0)
        # since this is a willingly written code we ourselves removing a req from the
        # batch and cover the loop again
        #  the loop runs until there are no more requests in the batch and till the queue gets empty

                             #    remove completed request immediately

Processing batch of 4 requests simultaneously
Processing batch of 4 requests simultaneously
Processing batch of 4 requests simultaneously
Processing batch of 4 requests simultaneously
Processing batch of 4 requests simultaneously
Processing batch of 4 requests simultaneously
Processing batch of 4 requests simultaneously
Processing batch of 3 requests simultaneously
Processing batch of 2 requests simultaneously
Processing batch of 1 requests simultaneously
